In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("../data/mutual_funds_cleaned.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/mutual_funds_cleaned.csv'

In [3]:
import os

os.listdir("../data")

['processed', 'raw']

In [4]:
os.listdir("../data/processed")

[]

In [5]:
os.listdir("../data/raw")

['01_fund_master.csv',
 '02_nav_history.csv',
 '03_aum_by_fund_house.csv',
 '04_monthly_sip_inflows.csv',
 '05_category_inflows.csv',
 '06_industry_folio_count.csv',
 '07_scheme_performance.csv',
 '08_investor_transactions.csv',
 '09_portfolio_holdings.csv',
 '10_benchmark_indices.csv']

In [6]:
import pandas as pd

df = pd.read_csv("../data/raw/01_fund_master.csv")
df.head()

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [7]:
features = ['category', 'sub_category', 'fund_house', 'plan']

df_rec = df[features + ['scheme_name']].copy()

df_rec.head()

,category,sub_category,fund_house,plan,scheme_name
0,Equity,Large Cap,SBI Mutual Fund,Regular,SBI Bluechip Fund - Regular Plan - Growth
1,Equity,Large Cap,SBI Mutual Fund,Direct,SBI Bluechip Fund - Direct Plan - Growth
2,Equity,Small Cap,SBI Mutual Fund,Regular,SBI Small Cap Fund - Regular Plan - Growth
3,Equity,Small Cap,SBI Mutual Fund,Direct,SBI Small Cap Fund - Direct Plan - Growth
4,Debt,Gilt,SBI Mutual Fund,Regular,SBI Magnum Gilt Fund - Regular Plan - Growth


In [8]:
df_rec['combined_features'] = (
    df_rec['category'].astype(str) + ' ' +
    df_rec['sub_category'].astype(str) + ' ' +
    df_rec['fund_house'].astype(str) + ' ' +
    df_rec['plan'].astype(str)
)

df_rec[['scheme_name', 'combined_features']].head()

,scheme_name,combined_features
0,SBI Bluechip Fund - Regular Plan - Growth,Equity Large Cap SBI Mutual Fund Regular
1,SBI Bluechip Fund - Direct Plan - Growth,Equity Large Cap SBI Mutual Fund Direct
2,SBI Small Cap Fund - Regular Plan - Growth,Equity Small Cap SBI Mutual Fund Regular
3,SBI Small Cap Fund - Direct Plan - Growth,Equity Small Cap SBI Mutual Fund Direct
4,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt Gilt SBI Mutual Fund Regular


In [9]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()

count_matrix = cv.fit_transform(df_rec['combined_features'])

count_matrix.shape

(40, 37)

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(count_matrix)

similarity.shape

(40, 40)

In [11]:
def recommend_funds(fund_name):

    idx = df_rec[df_rec['scheme_name'] == fund_name].index[0]

    scores = list(enumerate(similarity[idx]))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    scores = scores[1:6]

    for i in scores:
        print(df_rec.iloc[i[0]]['scheme_name'])

In [12]:
recommend_funds("SBI Bluechip Fund - Regular Plan - Growth")

SBI Bluechip Fund - Direct Plan - Growth
SBI Small Cap Fund - Regular Plan - Growth
HDFC Top 100 Fund - Regular Plan - Growth
Axis Bluechip Fund - Regular - Growth
DSP Top 100 Equity Fund - Regular - Growth
